<p style="font-family: Cambria; text-align: center; font-size: 48px;"> Responsivenes DATA Cleaning</p>

<h2>Overview</h2>
<p style="font-family: Cambria; font-size: 16px;"><b>The Responsiveness dataset contains information about patients' neurological responsiveness using the Glasgow Coma Scale (GCS).
<ul><li>Total Records: 2,008 patients</li>

<li>Total Columns: 6</li>

<li>Key Measures: Eye Opening, Verbal Response, Movement, Consciousness, and GCS</li>

<li>GCS Range: 3–15</li>

<li>Data Quality: No missing values, duplicate records, or duplicate patient IDs</li>
<li>Key Insight: Around 97% of patients have a GCS score of 15, and approximately 98% are classified as Clear, indicating that most patients in the dataset show normal responsiveness.</li>
<li>Cleaning Focus: Validate GCS ranges, standardize categories, verify GCS calculations, and perform final data-quality checks.</li>

</ul>

In [26]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

In [27]:
import warnings
warnings.simplefilter("ignore", UserWarning)

In [28]:
# Define the main project folder
project_folder = Path(
    r'C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure'
)

# Define the folder containing the original datasets
raw_data_folder = project_folder / "raw_data"

# Define the folder for cleaned datasets and reports
output_folder = project_folder / "cleaned_data"

# Create the output folder if it does not already exist
output_folder.mkdir(exist_ok=True)

print("Project Folder:", project_folder)
print("Raw Data Folder:", raw_data_folder)
print("Output Folder:", output_folder) 


Project Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure
Raw Data Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\raw_data
Output Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\cleaned_data


In [29]:
# Read the original Labs dataset
df = pd.read_csv(raw_data_folder / "responsivenes.csv")

# Keep an original copy
df_original = df.copy()

# Display the first five records
display(df.head())

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()
df.info()
df.describe()

,inpatient_number,eye_opening,verbal_response,movement,consciousness,gcs
0,857781,4,5,6,Clear,15
1,743087,4,5,6,Clear,15
2,866418,4,5,6,Clear,15
3,775928,4,5,6,Clear,15
4,810128,4,5,6,Clear,15


Rows: 2008
Columns: 6
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   inpatient_number  2008 non-null   int64 
 1   eye_opening       2008 non-null   int64 
 2   verbal_response   2008 non-null   int64 
 3   movement          2008 non-null   int64 
 4   consciousness     2008 non-null   object
 5   gcs               2008 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 94.3+ KB


,inpatient_number,eye_opening,verbal_response,movement,gcs
count,2008.000000,2008.000000,2008.000000,2008.000000,2008.000000
mean,797747.542829,3.963645,4.940239,5.927291,14.831175
std,41127.801740,0.282654,0.426354,0.548306,1.179836
min,722128.000000,1.000000,1.000000,1.000000,3.000000
25%,763164.500000,4.000000,5.000000,6.000000,15.000000
50%,798758.000000,4.000000,5.000000,6.000000,15.000000
75%,829399.750000,4.000000,5.000000,6.000000,15.000000
max,905720.000000,4.000000,5.000000,6.000000,15.000000


<h3>1. Check Data Types</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason: </b>To make sure numerical and categorical columns have the correct data type.</p>

In [30]:
df.dtypes

inpatient_number     int64
eye_opening          int64
verbal_response      int64
movement             int64
consciousness       object
gcs                  int64
dtype: object

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight: </b>weight, height, and bmi should be numeric, while gender, occupation, and agecat are categorical.</p>

<h3>2. Check Missing Values</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify missing values in each column before cleaning the dataset.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason :</b> Missing GCS components or consciousness information could make neurological assessment incomplete or unreliable.</p>

In [31]:
df.isnull().sum()

inpatient_number    0
eye_opening         0
verbal_response     0
movement            0
consciousness       0
gcs                 0
dtype: int64

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Our dataset has 0 missing values in all 6 columns.</p>
<p style="font-family: Cambria; font-size: 16px;"><b>Decision : </b>Do not remove or fill any records. No missing-value treatment is required.</p>

<h3>3. Check Duplicate Records</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We identify duplicate rows that could result in repeated information.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Duplicate records can artificially increase patient counts and bias later analysis.</p>

In [32]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>There are 0 completely duplicated rows.</p>


<h3>4. Check Duplicate Patient IDs</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We verify whether the same patient ID appears more than once.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Even when entire rows are not duplicated, the same patient ID might accidentally appear multiple times.</p>

In [33]:
print(
    "Duplicate Patient IDs:",
    df["inpatient_number"].duplicated().sum()
)

print(
    "Unique Patients:",
    df["inpatient_number"].nunique()
)

Duplicate Patient IDs: 0
Unique Patients: 2008


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>There are 0 completely duplicated rows.</p>

<h3>5. Standardize Column Names</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We remove unnecessary spaces from categorical columns to make values consistent.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Standard column names reduce coding errors and make the dataset easier to work with.</p>

In [34]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns

Index(['inpatient_number', 'eye_opening', 'verbal_response', 'movement',
       'consciousness', 'gcs'],
      dtype='object')

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight </b>Your existing column names are already well structured, so this is mainly a preventive cleaning step.</p>

<h3>6. Clean the Consciousness Category</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>ResponsiveToSound and ResponsiveToPain are valid categories, but adding spaces makes them more readable in charts, dashboards, and reports.</p>

In [35]:
df["consciousness"] = (
    df["consciousness"]
      .str.strip()
      .replace({
          "ResponsiveToSound": "Responsive To Sound",
          "ResponsiveToPain": "Responsive To Pain"
      })
)

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>98.31% of records are classified as Clear, while only a small proportion show reduced responsiveness.</p>

<h3>7. Validate Eye Opening Range</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Eye opening is a GCS component and should remain within its expected coded range.</p>

In [36]:
df["eye_opening"].value_counts().sort_index()

eye_opening
1      14
2       3
3      25
4    1966
Name: count, dtype: int64

In [37]:
#Check invalid values:
invalid_eye = df[
    ~df["eye_opening"].between(1, 4)
]

print("Invalid Eye Opening:", len(invalid_eye))

Invalid Eye Opening: 0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Our observed values range from 1 to 4, so there are no obvious out-of-range values.</p>

<h3>8. Validate Verbal Response</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Values outside the expected 1–5 coding range could indicate data-entry errors.</p>

In [38]:
invalid_verbal = df[
    ~df["verbal_response"].between(1, 5)
]

print("Invalid Verbal Response:", len(invalid_verbal))

Invalid Verbal Response: 0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Our verbal-response values range from 1 to 5, so they pass the range check.</p>

<h3>9. Validate Movement</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Movement/motor response should follow its expected GCS component range.</p>

In [39]:
invalid_movement = df[
    ~df["movement"].between(1, 6)
]

print("Invalid Movement:", len(invalid_movement))

Invalid Movement: 0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Our movement values range from 1 to 6, so no out-of-range records were found.</p>

<h3>10. Validate GCS Range</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>The total Glasgow Coma Scale is formed from the three component scores and should fall between 3 and 15.</p>

invalid_gcs = df[
    ~df["gcs"].between(3, 15)
]

print("Invalid GCS:", len(invalid_gcs))

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>Our GCS values range from 3 to 15, so all records fall within the expected total range.</p>

<h3>11. Most Important Quality Check — Recalculate GCS</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>This is especially important for this dataset because:
GCS = Eye Opening + Verbal Response + Movement</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>A GCS value could be within 3–15 but still be incorrect if it does not equal the sum of its three components.</p>

In [40]:
#Create a temporary calculated column:
df["calculated_gcs"] = (
    df["eye_opening"]
    + df["verbal_response"]
    + df["movement"]
)

In [41]:
#Compare it with the existing gcs:
gcs_mismatch = df[
    df["gcs"] != df["calculated_gcs"]
]

print("GCS Mismatches:", len(gcs_mismatch))

GCS Mismatches: 0


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>0 mismatches were found. All 2,008 records satisfy. GCS = Eye Opening + Verbal Response + Movement<p>
This is a strong data-quality result.</p>

<h3>12. Remove the Temporary Validation Column</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>calculated_gcs was created only for validation. Keeping it would duplicate information already stored in gcs.</p>

In [42]:
df.drop(
    columns=["calculated_gcs"],
    inplace=True
)

In [43]:
df

,inpatient_number,eye_opening,verbal_response,movement,consciousness,gcs
0,857781,4,5,6,Clear,15
1,743087,4,5,6,Clear,15
2,866418,4,5,6,Clear,15
3,775928,4,5,6,Clear,15
4,810128,4,5,6,Clear,15
...,...,...,...,...,...,...
2003,740689,4,5,6,Clear,15
2004,734280,4,5,6,Clear,15
2005,781004,4,5,6,Clear,15
2006,744870,1,1,1,Nonresponsive,3


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>No original data is lost; only the temporary validation field is removed.</p>

<h3>13. Review GCS Distribution</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>Distribution checks can reveal suspicious values or unexpected patterns that basic range checks cannot detect.</p>

In [44]:
df["gcs"].value_counts().sort_index()

gcs
3       13
4        1
6        1
7        4
10       7
11      19
12       3
13       2
14       7
15    1951
Name: count, dtype: int64

<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight</b>1,951 of 2,008 records (about 97.16%) have GCS = 15, so the dataset is strongly concentrated at the maximum GCS score. This is not automatically an error, it is an important characteristic of the dataset and should be considered when doing later analysis or modeling.</p>

<h3>14. Final Data Quality Check</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To confirm the cleaning process worked.</p>

In [45]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Patient IDs:",
    df["inpatient_number"].duplicated().sum()
)

print("\nGCS Range:")
print(df["gcs"].min(), "-", df["gcs"].max())

print("\nConsciousness Distribution:")
print(df["consciousness"].value_counts())

Rows: 2008
Columns: 6

Missing Values:
inpatient_number    0
eye_opening         0
verbal_response     0
movement            0
consciousness       0
gcs                 0
dtype: int64

Duplicate Rows: 0
Duplicate Patient IDs: 0

GCS Range:
3 - 15

Consciousness Distribution:
consciousness
Clear                  1974
Responsive To Sound      19
Nonresponsive            11
Responsive To Pain        4
Name: count, dtype: int64


<h3>15. Save Cleaned Dataset</h3>
<p style="font-family: Cambria; font-size: 16px;"><b>We save the cleaned dataset as a new CSV file for further analysis.</b></p>
<p style="font-family: Cambria; font-size: 16px;"><b>Reason : </b>To save the cleaned dataset as a new CSV file so the original dataset remains unchanged and the cleaned version can be used for further analysis.</p>

In [46]:
# Save the cleaned dataset and supporting reports

# Create the output folder if it does not exist
output_folder.mkdir(exist_ok=True)

# Save cleaned dataset
df.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_DataCleaning_responsivenes.csv",
    index=False
)

# Create missing-value report
missing_report = df.isnull().sum().reset_index()
missing_report.columns = ['Column', 'Missing_Count']

# Save missing-value report
missing_report.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_Missing_Value_Report_responsivenes.csv",
    index=False
)

# Create flagged records
flagged_records = df[df.isnull().any(axis=1)].copy()

# Save flagged records
flagged_records.to_csv(
    output_folder / "Team7_CodeAvengers_Category1_Flagged_Records_responsivenes.csv",
    index=False
)

# Display the saved files and their locations
print("All files saved successfully!\n")

for file in output_folder.iterdir():
    if file.is_file():
        print("Saved:", file.name)

print("\nOutput Folder:", output_folder)
print("Final Cleaned Dataset Shape:", df.shape)

All files saved successfully!

Saved: demography_cleaned_simple.csv
Saved: Team7_CodeAvengers_Category1_DataCleaning_demography
Saved: Team7_CodeAvengers_Category1_DataCleaning_demography.csv
Saved: Team7_CodeAvengers_Category1_DataCleaning_responsivenes.csv
Saved: Team7_CodeAvengers_Category1_Flagged_Records_demography.csv
Saved: Team7_CodeAvengers_Category1_Flagged_Records_responsivenes.csv
Saved: Team7_CodeAvengers_Category1_Missing_Value_Report_demography.csv
Saved: Team7_CodeAvengers_Category1_Missing_Value_Report_responsivenes.csv

Output Folder: C:\Neha Career\Numpy Ninja\Python\Python Hackthon\Hackthon 2026\Python_Hackathon_Sep_2026\cardiac_failure\cleaned_data
Final Cleaned Dataset Shape: (2008, 6)


<p style="font-family: Cambria; font-size: 16px;"><b>Actual Insight : </b>The cleaned dataset now contains the records retained after handling incomplete records, duplicates, invalid/incomplete measurements, and unnecessary temporary columns. Saving it separately preserves the original raw dataset for reference while providing an analysis-ready dataset.</p>

<h3>Final cleaning conclusion</h3>

<p style="font-family: Cambria; font-size: 16px;">Your responsivenes.csv is a high-quality dataset that requires relatively little destructive cleaning. The main findings are:

<ul><li>2,008 records and 6 columns</li>
<li>0 missing values</li>
<li>0 duplicate rows</li>
<li>0 duplicate inpatient IDs</li>
<li>Eye opening values are within 1–4</li>
<li>Verbal response values are within 1–5</li>
<li>Movement values are within 1–6</li>
<li>GCS values are within 3–15</li>
<li>0 GCS calculation mismatches</li>
<li>98.31% of patients are recorded as Clear</li>
<li>About 97.16% have a GCS score of 15</li></ul>